In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set clean styling
sns.set_theme(style="whitegrid")

# Load your verified, clean Parquet data
df = pd.read_parquet("data/processed/healthcare_clean.parquet")
print(f"Dataset Loaded Successfully! Records: {len(df):,}")
df.info()

In [ ]:
# 1. Summary table by Medical Condition
los_summary = df.groupby("Medical Condition")["Length_of_Stay_Days"].agg(
    Total_Patients="count",
    Mean_LOS="mean",
    Median_LOS="median",
    P90_LOS=lambda x: np.percentile(x, 90),
    Max_LOS="max"
).round(2).sort_values(by="Mean_LOS", ascending=False)

display(los_summary)

# 2. Distribution Plot
plt.figure(figsize=(10, 4.5))
sns.boxplot(
    data=df, 
    x="Medical Condition", 
    y="Length_of_Stay_Days", 
    palette="mako", 
    hue="Medical Condition", 
    legend=False
)
plt.title("Length of Stay (LOS) Dispersion Across Clinical Conditions", fontsize=12, fontweight="bold")
plt.xlabel("Medical Condition", fontsize=10)
plt.ylabel("Stay Duration (Days)", fontsize=10)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
# Aggregate daily billing rate across Insurance Providers and Admission Types
billing_pivot = df.pivot_table(
    index="Insurance Provider",
    columns="Admission Type",
    values="Daily_Billing_Rate",
    aggfunc="median"
).round(2)

display(billing_pivot)

# Heatmap Visualization
plt.figure(figsize=(8, 4))
sns.heatmap(billing_pivot, annot=True, fmt=".2f", cmap="YlGnBu", cbar_kws={'label': 'Median Daily Rate ($)'})
plt.title("Median Daily Billing Rate by Insurance Provider & Admission Severity", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# 1. Medication protocol matrix by Medical Condition
med_matrix = pd.crosstab(
    df["Medical Condition"], 
    df["Medication"], 
    normalize="index"
).round(4) * 100

print("Medication Distribution (% within Condition):")
display(med_matrix)

# 2. Billing impact by Test Result
test_billing = df.groupby(["Medical Condition", "Test Results"])["Billing Amount"].agg(
    Avg_Bill="mean",
    Median_Bill="median"
).round(2).unstack()

print("\nBilling Amount by Condition & Diagnostic Test Result:")
display(test_billing)

In [ ]:
# Cell 5: Executive Metrics Export
executive_summary = df.groupby(["Medical Condition", "Admission Type"]).agg(
    Patient_Volume=("Patient_ID", "count"),
    Avg_Stay_Days=("Length_of_Stay_Days", "mean"),
    Median_Daily_Cost=("Daily_Billing_Rate", "median"),
    Total_Revenue=("Billing Amount", "sum")
).round(2)

# Save summary metrics to parquet for downstream reporting
executive_summary.to_parquet("data/processed/executive_operational_summary.parquet")
print("Executive operational metrics serialized to 'data/processed/executive_operational_summary.parquet'!")
display(executive_summary.head(10))